# Purchases Costing & SKU Matching Agent

**Objective:** To take a raw purchases CSV, clean and standardize the product names, parse unstructured quantity data, and match each item to an approved SKU from the database.

**Workflow:** This notebook uses an interactive, multi-phase process. At each step where AI assistance is used, the script will:
1.  **Propose Changes:** Use the Gemini API to suggest standardized names, parse data, or match SKUs.
2.  **Export for Review:** Save the proposals to an Excel file.
3.  **Await Validation:** Pause and allow you to open the Excel file, review the AI's suggestions, make corrections, and save the file.
4.  **Load & Continue:** Use your validated file as the input for the next phase.

This "bounce-back" method ensures you have full control and can validate the data at every critical step.


## Phase 0: Instructions & Setup

---
### **Workflow Rules: Please Read Before Running**

**This notebook is interactive. Follow these rules to avoid errors and unnecessary API calls.**

#### A) First Time Running or After Restarting:
1.  **Run ALL cells in order from top to bottom.** This is required to load data and build the variables needed for later steps.

#### B) After Changing a `.py` File in the `/src` Directory:
1.  **Do NOT restart the kernel.**
2.  **Run ONLY the "Phase 0" Setup Cell below.** (The one with `%autoreload 2`). This loads your changes.
3.  **SKIP to the specific Phase you want to test** (e.g., Phase 2 or 4) and run that cell directly. This avoids re-running earlier phases and saves API costs.

---


In [1]:
# This cell enables autoreload. Run it once when you start, and re-run it
# anytime you change a .py file in the /src directory.
%load_ext autoreload
%autoreload 2

# This allows us to import from the /src directory
import sys
import os
import pandas as pd

# --- Path Setup ---
# Add the project root to the Python path
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added '{module_path}' to the Python path.")

# --- Import Custom Modules ---
# Now you can import your modules
# from src.utils import some_function
# from src.processing import another_function
    
# --- Configuration ---
# Define the data directories relative to the notebook's location
DATA_DIR = os.path.join('..', 'data')
INPUT_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DIR = os.path.join(DATA_DIR, 'output')

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Setup complete. Autoreload is active.")
print(f"Input directory set to: {os.path.abspath(INPUT_DIR)}")
print(f"Output directory set to: {os.path.abspath(OUTPUT_DIR)}")


Added 'c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3' to the Python path.
✅ Setup complete. Autoreload is active.
Input directory set to: c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\recipe_analyzer\data\input
Output directory set to: c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\recipe_analyzer\data\output


## Phase 1: Load & Initial Cleaning

**Goal:** Load the raw purchases CSV file and perform initial cleaning.
-   Standardize column names to lowercase `snake_case`.
-   Create a `product_normalized` column for easier matching later.
-   Display the first few rows to verify the data is loaded correctly.


In [2]:
# --- Configuration ---
INPUT_FILE = os.path.join(INPUT_DIR, '01_CUATRO PATAS_2026_FEB - COMPRAS.csv')

# --- Action ---
print("Purchases Costing Agent: Initializing...")
print(f"Attempting to load data from: {INPUT_FILE}")

try:
    df = pd.read_csv(INPUT_FILE)
    
    # --- Basic Cleaning ---
    # 1. Standardize column names (snake_case, lowercase)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
    
    # 2. Create a normalized product name for matching
    if 'producto' in df.columns:
        df['product_normalized'] = df['producto'].str.strip().str.lower()
        print("✅ Created 'product_normalized' column.")
    else:
        print("⚠️ WARNING: 'producto' column not found.")

    print("\n✅ Data loaded and cleaned successfully.")
    print(f"Found {df.shape[0]} rows and {df.shape[1]} columns.")
    print("\nHere is a preview of your data:")
    display(df.head())
    
except FileNotFoundError:
    print(f"❌ ERROR: File not found at '{INPUT_FILE}'. Please ensure the file exists.")
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred: {e}")



Purchases Costing Agent: Initializing...
Attempting to load data from: ..\data\input\01_CUATRO PATAS_2026_FEB - COMPRAS.csv
✅ Created 'product_normalized' column.

✅ Data loaded and cleaned successfully.
Found 61 rows and 4 columns.

Here is a preview of your data:


,categoria,producto,cantidad,product_normalized
0,PROTEINAS,OSTIONES VIVOS,280PZ,ostiones vivos
1,PROTEINAS,TOCINO AHUMADO,150G,tocino ahumado
2,PROTEINAS,LOMOS DE JUREL,1PZ (8-9KG),lomos de jurel
3,PROTEINAS,HOMBRO DE CERDO,2PZ (8-10KG),hombro de cerdo
4,ABARROTES,MISO ROJO (AKA MISO),300G,miso rojo (aka miso)


## Phase 2: AI-Powered Duplicate Consolidation

**Goal:** Identify similar product names and propose a single "canonical" name for them.
-   Gathers all unique product names from the `product_normalized` column.
-   Sends the list to the Gemini API and asks it to propose a standardized name for each.
-   Adds the AI's suggestions to a new `ai_proposed_canonical_name` column.
-   Creates an empty `product_canonical` column for you to fill in.
-   Exports the result to `01_purchases_for_canonical_review.xlsx`.

**ACTION REQUIRED:**
1.  **Run this cell.**
2.  **Open the generated Excel file:** `recipe_analyzer/data/output/01_purchases_for_canonical_review.xlsx`.
3.  **Review the proposals:** Check the `ai_proposed_canonical_name` column.
4.  **Validate or Correct:** Copy the good proposals into the `product_canonical` column. For bad proposals, manually enter the correct standard name in that column.
5.  **Save the file.** You will load this validated file in the next phase.


In [3]:
import json
from recipe_analyzer.ai_utils import configure_gemini_api, create_canonical_name_prompt, call_gemini_api, extract_json_from_string

# --- 1. Configure API ---
# This only needs to be done once.
try:
    configure_gemini_api()
except ValueError as e:
    print(f"❌ CONFIGURATION ERROR: {e}")
    # Stop execution if API key is not found
    raise

# --- 2. Gather Context for the AI ---
print("\n--- 1. Gathering Unique Products for AI ---")
if 'product_normalized' not in df.columns:
    raise KeyError("The required column 'product_normalized' was not found. Please run Phase 1 first.")
    
unique_products = df['product_normalized'].dropna().unique().tolist()
print(f"Found {len(unique_products)} unique products to classify.")

# --- 3. AI Classification Proposal ---
print("\n--- 2. Getting AI Proposal for Canonical Names ---")
prompt = create_canonical_name_prompt(unique_products)
ai_response_str = call_gemini_api(prompt)

# --- 4. Parse and Apply AI Suggestions ---
print("\n--- 3. Parsing and Applying AI Suggestions ---")
product_mappings = extract_json_from_string(ai_response_str)

if product_mappings:
    df['ai_proposed_canonical_name'] = df['product_normalized'].map(product_mappings)
    
    # Create the column for user validation
    df['product_canonical'] = '' # Leave it blank for the user to fill
    
    print("✅ Successfully applied AI suggestions.")
    print("Here's a preview of the proposals:")
    display(df[['product_normalized', 'ai_proposed_canonical_name']].drop_duplicates().head())
else:
    print("❌ ERROR: Could not parse a valid JSON mapping from the AI response. Cannot proceed with applying suggestions.")
    print(f"Raw Response was:\n---\n{ai_response_str}\n---")

# --- 5. Export for Human Review ---
if product_mappings: # Only export if the AI part was successful
    REVIEW_FILE_PATH = os.path.join(OUTPUT_DIR, '01_purchases_for_canonical_review.xlsx')
    try:
        # Define the column order for the output file
        export_columns = [
            'product_canonical', # Put user column first
            'ai_proposed_canonical_name',
            'producto',
            'product_normalized',
            'cantidad',
            'unidad',
            'costo_unitario',
            'costo_total'
        ]
        # Filter for columns that actually exist in the dataframe
        export_columns = [col for col in export_columns if col in df.columns]

        df[export_columns].to_excel(REVIEW_FILE_PATH, index=False)
        print(f"\n✅ Successfully exported the data for your review.")
        print(f"ACTION REQUIRED: Open the file below, validate/correct the `product_canonical` column, and save.")
        print(f"FILE -> {os.path.abspath(REVIEW_FILE_PATH)}")
    except Exception as e:
        print(f"❌ ERROR: An unexpected error occurred during export: {e}")



✅ Gemini API configured successfully.

--- 1. Gathering Unique Products for AI ---
Found 61 unique products to classify.

--- 2. Getting AI Proposal for Canonical Names ---
✅ Gemini model 'gemini-1.5-flash-latest' initialized successfully.
Submitting prompt to Gemini API...
❌ ERROR: An error occurred while calling the Gemini API: 404 models/gemini-1.5-flash-latest is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

--- 3. Parsing and Applying AI Suggestions ---
✅ Successfully applied AI suggestions.
Here's a preview of the proposals:


,product_normalized,ai_proposed_canonical_name
0,ostiones vivos,NaN
1,tocino ahumado,NaN
2,lomos de jurel,NaN
3,hombro de cerdo,NaN
4,miso rojo (aka miso),NaN



✅ Successfully exported the data for your review.
ACTION REQUIRED: Open the file below, validate/correct the `product_canonical` column, and save.
FILE -> c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\recipe_analyzer\data\output\01_purchases_for_canonical_review.xlsx


## Phase 3: AI-Powered Quantity & Unit Parsing

**Goal:** Load the validated canonical names and then parse the unstructured `cantidad` column into structured data (`value`, `unit`, `notes`).
-   Loads the Excel file you saved in the previous step.
-   Verifies that you have filled in the `product_canonical` column.
-   Sends the unique quantity strings to the Gemini API for parsing.
-   Adds the AI's proposals to `ai_proposed_value`, `ai_proposed_unit`, and `ai_proposed_notes`.
-   Creates empty columns (`quantity_value`, `quantity_unit`, `quantity_notes`) for your validation.
-   Exports the result to `02_purchases_for_quantity_review.xlsx`.

**ACTION REQUIRED:**
1.  **Run this cell.**
2.  **Open the generated Excel file:** `recipe_analyzer/data/output/02_purchases_for_quantity_review.xlsx`.
3.  **Review the proposals:** Check the AI-proposed quantity columns.
4.  **Validate or Correct:** Copy the correct values into the final `quantity_value`, `quantity_unit`, and `quantity_notes` columns.
5.  **Save the file.** You will load this validated file in the next phase.


In [5]:
import time
import numpy as np
from recipe_analyzer.ai_utils import create_quantity_parser_prompt, call_gemini_api, extract_json_from_string

# --- 1. Load the reviewed file ---
CANONICAL_REVIEW_FILE = os.path.join(OUTPUT_DIR, '01_purchases_for_canonical_review.xlsx')
print(f"--- 1. Loading your reviewed file from: {CANONICAL_REVIEW_FILE} ---")
try:
    df_canonical = pd.read_excel(CANONICAL_REVIEW_FILE)
    
    # --- Data Validation ---
    if 'product_canonical' not in df_canonical.columns or df_canonical['product_canonical'].isnull().all():
        raise ValueError("The 'product_canonical' column is empty. Please fill it in before proceeding.")
    
    # Forward-fill the canonical name for easier processing
    df_canonical['product_canonical'].ffill(inplace=True)
    print("✅ Successfully loaded and validated the canonical names file.")
    
except FileNotFoundError:
    print(f"❌ ERROR: The file '{CANONICAL_REVIEW_FILE}' was not found. Please run Phase 2 first.")
    raise
except Exception as e:
    print(f"❌ ERROR: Could not load the reviewed file: {e}")
    raise

# --- 2. Batch Processing for AI Standardization ---
print("\n--- 2. Getting AI Quantity Parsing Proposal in Batches ---")
if 'cantidad' not in df_canonical.columns:
    raise KeyError("The required column 'cantidad' was not found.")
    
unique_quantities = df_canonical['cantidad'].dropna().unique().tolist()
quantity_mappings = {}
CHUNK_SIZE = 50  # Process 50 ingredients at a time
num_chunks = (len(unique_quantities) + CHUNK_SIZE - 1) // CHUNK_SIZE

for i in range(0, len(unique_quantities), CHUNK_SIZE):
    chunk = unique_quantities[i:i + CHUNK_SIZE]
    chunk_num = (i // CHUNK_SIZE) + 1
    print(f"\nProcessing chunk {chunk_num} of {num_chunks}...")
    
    prompt = create_quantity_parser_prompt(chunk)
    ai_response_str = call_gemini_api(prompt)
    
    chunk_mapping = extract_json_from_string(ai_response_str)
    
    if chunk_mapping:
        quantity_mappings.update(chunk_mapping)
        print(f"✅ Successfully processed chunk {chunk_num}.")
    else:
        print(f"⚠️ WARNING: Failed to process chunk {chunk_num}. The AI response was not valid JSON.")
        print(f"Raw Response for this chunk:\n---\n{ai_response_str}\n---")
    
    time.sleep(1) # Be respectful of API rate limits

print(f"\n✅ Finished processing all chunks. {len(quantity_mappings)} mappings were created.")

# --- 3. Apply the combined mappings to the DataFrame ---
print("\n--- 3. Applying Full AI Proposal to DataFrame ---")
try:
    if not quantity_mappings:
        raise ValueError("The AI processing resulted in no valid quantity mappings.")

    # Create a temporary DataFrame from the mappings
    mappings_df = pd.DataFrame.from_dict(quantity_mappings, orient='index')
    mappings_df.index.name = 'cantidad'
    mappings_df = mappings_df.rename(columns={'value': 'ai_proposed_value', 'unit': 'ai_proposed_unit', 'notes': 'ai_proposed_notes'})
    
    # Merge the proposals back into the main DataFrame
    df_quantities = df_canonical.merge(mappings_df, on='cantidad', how='left')
    
    # Create empty columns for user validation
    df_quantities['quantity_value'] = np.nan
    df_quantities['quantity_unit'] = ''
    df_quantities['quantity_notes'] = ''

    print("✅ New 'ai_proposed_*' quantity columns added.")
    display(df_quantities[['cantidad', 'ai_proposed_value', 'ai_proposed_unit', 'ai_proposed_notes']].head())
    
except Exception as e:
    print(f"❌ ERROR: Could not apply AI response. Details: {e}")
    raise

# --- 4. Export the Result for Review ---
print("\n--- 4. Exporting for Quantity Review ---")
QUANTITY_REVIEW_FILE = os.path.join(OUTPUT_DIR, '02_purchases_for_quantity_review.xlsx')
try:
    # Define the column order for the output file
    export_columns = [
        # User validation columns first
        'quantity_value',
        'quantity_unit',
        'quantity_notes',
        # AI proposal columns
        'ai_proposed_value',
        'ai_proposed_unit',
        'ai_proposed_notes',
        # Original data
        'cantidad',
        'product_canonical',
        'producto',
        'unidad',
        'costo_unitario',
        'costo_total'
    ]
    export_columns = [col for col in export_columns if col in df_quantities.columns]

    df_quantities[export_columns].to_excel(QUANTITY_REVIEW_FILE, index=False)
    
    print(f"✅ Successfully exported data for your quantity review.")
    print(f"ACTION REQUIRED: Open the file, review and fill in the `quantity_*` columns, and save.")
    print(f"FILE -> {os.path.abspath(QUANTITY_REVIEW_FILE)}")
    
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred while exporting: {e}")



--- 1. Loading your reviewed file from: ..\data\output\01_purchases_for_canonical_review.xlsx ---
❌ ERROR: Could not load the reviewed file: The 'product_canonical' column is empty. Please fill it in before proceeding.


ValueError: The 'product_canonical' column is empty. Please fill it in before proceeding.

## Phase 4: AI-Powered SKU Matching

**Goal:** Match each cleaned, canonical product name to an official `ApprovedSku` from the main database.
-   Loads the validated quantity file from the previous phase.
-   Connects to the production database and fetches a complete list of `ApprovedSku` records.
-   For each unique product, it asks the Gemini API to find the best match from the SKU list.
-   Adds the AI's proposal to the `ai_proposed_sku` column.
-   Creates an empty `assigned_sku_key` for your validation.
-   Exports the result to `03_purchases_for_sku_review.xlsx`.

**ACTION REQUIRED:**
1.  **Run this cell.**
2.  **Open the generated Excel file:** `recipe_analyzer/data/output/03_purchases_for_sku_review.xlsx`.
3.  **Review the proposals:** Check the `ai_proposed_sku` against the `product_canonical`.
4.  **Validate or Correct:** Copy the correct SKU key into the `assigned_sku_key` column. If the AI couldn't find a match (`no_match_found`), you may need to look up the correct one manually.
5.  **Save the file.** This is the final validation step.


In [4]:
from recipe_analyzer.database_utils import get_all_approved_skus_df
from recipe_analyzer.ai_utils import create_sku_matching_prompt, call_gemini_api, extract_json_from_string
import time

# --- Configuration ---
# IMPORTANT: Set this to the client RFC you are working with.
# This ensures you only get SKUs relevant to that client.
CLIENT_RFC_TO_USE = "CUIK88060136A" # Using the example from .env.example

# --- 1. Load the reviewed file ---
QUANTITY_REVIEW_FILE = os.path.join(OUTPUT_DIR, '02_purchases_for_quantity_review.xlsx')
print(f"--- 1. Loading your reviewed file from: {QUANTITY_REVIEW_FILE} ---")
try:
    df_sku_matching = pd.read_excel(QUANTITY_REVIEW_FILE)
    
    # --- Data Validation ---
    if 'quantity_value' not in df_sku_matching.columns or df_sku_matching['quantity_value'].isnull().all():
        raise ValueError("The 'quantity_value' column is empty. Please fill it in before proceeding.")
    print("✅ Successfully loaded and validated the quantity review file.")

except FileNotFoundError:
    print(f"❌ ERROR: The file '{QUANTITY_REVIEW_FILE}' was not found. Please run Phase 3 first.")
    raise
except Exception as e:
    print(f"❌ ERROR: Could not load the reviewed file: {e}")
    raise
    
# --- 2. Fetch the Approved SKUs from the database ---
print("\n--- 2. Fetching Approved SKUs from Main Database ---")
approved_skus_df = get_all_approved_skus_df(client_rfc=CLIENT_RFC_TO_USE)

if approved_skus_df.empty:
    print("❌ ERROR: Failed to load approved SKUs from the database. Cannot proceed with matching.")
    raise ValueError("No Approved SKUs found.")

# --- 3. AI-Powered SKU Matching in Batches ---
print("\n--- 3. Matching Canonical Products to SKUs using AI ---")
if 'product_canonical' not in df_sku_matching.columns:
    raise KeyError("The required column 'product_canonical' was not found.")
    
unique_products_to_match = df_sku_matching['product_canonical'].dropna().unique()
sku_matches = {}

# Prepare a string representation of the SKU list for the AI's context
# Use the most relevant columns for matching.
sku_context_columns = ['sku_key', 'normalized_description', 'sub_sub_category', 'standardized_unit']
sku_list_string = approved_skus_df[sku_context_columns].to_string(index=False)

for product in unique_products_to_match:
    print(f"Processing product: '{product}'...")
    
    prompt = create_sku_matching_prompt(product, sku_list_string)
    ai_response_str = call_gemini_api(prompt)
    
    match_json = extract_json_from_string(ai_response_str)
    
    if match_json and 'best_match_sku' in match_json:
        sku_matches[product] = str(match_json['best_match_sku']).lower()
    else:
        sku_matches[product] = "no_match_found"
    
    time.sleep(1) # Be respectful of API rate limits

print("\n✅ Finished matching all products.")

# --- 4. Apply proposals and export for review ---
print("\n--- 4. Applying proposals and exporting for final review ---")
try:
    df_sku_matching['ai_proposed_sku'] = df_sku_matching['product_canonical'].map(sku_matches)
    df_sku_matching['assigned_sku_key'] = '' # For user validation

    SKU_REVIEW_FILE = os.path.join(OUTPUT_DIR, '03_purchases_for_sku_review.xlsx')
    
    # Define the column order for the output file
    export_columns = [
        # User validation columns first
        'assigned_sku_key',
        'ai_proposed_sku',
        # Validated data so far
        'product_canonical',
        'quantity_value',
        'quantity_unit',
        'quantity_notes',
        # Original data
        'producto',
        'cantidad',
        'costo_total'
    ]
    export_columns = [col for col in export_columns if col in df_sku_matching.columns]

    df_sku_matching[export_columns].to_excel(SKU_REVIEW_FILE, index=False)
    
    print(f"✅ Successfully exported data for your final SKU review.")
    print(f"ACTION REQUIRED: Open the file, review and fill in the `assigned_sku_key` column, and save.")
    print(f"FILE -> {os.path.abspath(SKU_REVIEW_FILE)}")
    
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred while applying suggestions or exporting: {e}")
    raise



[autoreload of recipe_analyzer.ai_utils failed: Traceback (most recent call last):
  File "c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\venv\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 524, in maybe_reload_module
    new_source_code = f.read()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2032.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 1456: character maps to <u

--- 1. Loading your reviewed file from: ..\data\output\02_purchases_for_quantity_review.xlsx ---
❌ ERROR: The file '..\data\output\02_purchases_for_quantity_review.xlsx' was not found. Please run Phase 3 first.


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\output\\02_purchases_for_quantity_review.xlsx'

## Phase 5: Load Final SKUs & Enrich Data

**Goal:** Load the final, human-validated data and enrich it by merging it with the full `ApprovedSku` dataset.
-   Loads the `03_purchases_for_sku_review.xlsx` file you just saved.
-   Verifies that you have filled in the `assigned_sku_key` column.
-   Performs a `merge` operation to join the purchase data with the corresponding SKU details (like category, subcategory, standardized_unit, etc.).
-   Displays the head of the final, fully-enriched DataFrame for a quick preview.


In [ ]:
# --- 1. Load the final reviewed file ---
SKU_REVIEW_FILE = os.path.join(OUTPUT_DIR, '03_purchases_for_sku_review.xlsx')
print(f"--- 1. Loading your final validated file from: {SKU_REVIEW_FILE} ---")
try:
    df_final = pd.read_excel(SKU_REVIEW_FILE)
    
    # --- Data Validation ---
    if 'assigned_sku_key' not in df_final.columns or df_final['assigned_sku_key'].isnull().all():
        raise ValueError("The 'assigned_sku_key' column is empty. Please fill it in before proceeding.")
    
    # Remove any rows where a match wasn't found or assigned
    df_final = df_final[df_final['assigned_sku_key'] != 'no_match_found'].dropna(subset=['assigned_sku_key'])

    print("✅ Successfully loaded and validated the final SKU review file.")

except FileNotFoundError:
    print(f"❌ ERROR: The file '{SKU_REVIEW_FILE}' was not found. Please run Phase 4 first.")
    raise
except Exception as e:
    print(f"❌ ERROR: Could not load the reviewed file: {e}")
    raise
    
# --- 2. Merge with Approved SKU Data ---
print("\n--- 2. Enriching data with full SKU details ---")
try:
    # Ensure the key in the approved_skus_df matches the format for merging
    # The key from the DB is 'sku_key', let's make sure it's lowercase.
    if 'sku_key' in approved_skus_df.columns:
         approved_skus_df['assigned_sku_key'] = approved_skus_df['sku_key'].str.lower()
    else:
        raise KeyError("'sku_key' not found in the approved SKUs DataFrame.")

    # Select only the columns we need from the SKU table to avoid clutter
    sku_details_to_merge = approved_skus_df[[
        'assigned_sku_key', 'category', 'subcategory', 'sub_sub_category',
        'standardized_unit', 'units_per_package', 'confidence_score'
    ]]

    # Perform the merge
    df_enriched = pd.merge(df_final, sku_details_to_merge, on='assigned_sku_key', how='left')

    print("✅ Successfully merged purchase data with SKU details.")
    print("Here is a preview of your final, enriched data:")
    display(df_enriched.head())

except Exception as e:
    print(f"❌ ERROR: An error occurred while merging SKU data: {e}")
    raise



## Phase 6: Final Export

**Goal:** Export the final, cleaned, and enriched dataset to a new Excel file.
-   This is the last step. It saves the completed DataFrame to `purchases_final_enriched.xlsx`.
-   This file contains your clean purchase records, linked to the correct SKUs, with standardized quantities and units, ready for further analysis or import into other systems.


In [ ]:
# --- Export the Final, Enriched Data ---
print("\n--- 5. Exporting final data with costs ---")
FINAL_ENRICHED_FILE = os.path.join(OUTPUT_DIR, 'purchases_final_enriched.xlsx')
try:
    df_enriched.to_excel(FINAL_ENRICHED_FILE, index=False)
    print(f"✅ FINAL SUCCESS! The fully enriched purchases file is saved at:")
    print(os.path.abspath(FINAL_ENRICHED_FILE))
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred while exporting the final file: {e}")

